# Análise de Sentimento v2 - Aplicando ao banco de dados do Kaggle

In [1]:
# Carregando pacotes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk import ngrams
from nltk.tokenize import word_tokenize
from collections import Counter
import spacy
nlp = spacy.load('pt_core_news_sm')
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\luans\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df = pd.read_csv('dados/financial_sentences_ptbr.csv')

df.head()

,sentence,sentiment,sentence_clear,sentence_pt
0,The GeoSolutions technology will leverage Bene...,positive,The GeoSolutions technology will leverage Bene...,A tecnologia GeoSolutions aproveitará as soluç...
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative,"ESI on lows, down 1.50 to 2.50 BK a real possi...","ESI em mínimas, queda de 1,50 a 2,50. BK é uma..."
2,"For the last quarter of 2010 , Componenta 's n...",positive,"For the last quarter of 2010 , Componenta 's n...","No último trimestre de 2010, as vendas líquida..."
3,According to the Finnish-Russian Chamber of Co...,neutral,According to the Finnish-Russian Chamber of Co...,"Segundo a Câmara de Comércio Finlandesa-Russa,..."
4,The Swedish buyout firm has sold its remaining...,neutral,The Swedish buyout firm has sold its remaining...,A empresa sueca de aquisições vendeu sua parti...


## Processamento de Texto

In [5]:
import string

stop_words = set(stopwords.words('portuguese'))
stop_words.update(['em', 'um', 'uma', 'a', 'as', 'o', 'os', 'de', 'do', 'da', 'dos', 'das', 'e', 'que', 'com', 'para', 'por', 'no', 'na', 'nos', 'nas'])

df['sentence_pt_clean'] = df['sentence_pt'].str.lower() 
df['sentence_pt_clean'] = df['sentence_pt_clean'].str.translate(str.maketrans('', '', string.punctuation))
df['sentence_pt_clean'] = df['sentence_pt_clean'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

df.head()

,sentence,sentiment,sentence_clear,sentence_pt,sentence_pt_clean
0,The GeoSolutions technology will leverage Bene...,positive,The GeoSolutions technology will leverage Bene...,A tecnologia GeoSolutions aproveitará as soluç...,tecnologia geosolutions aproveitará soluções g...
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative,"ESI on lows, down 1.50 to 2.50 BK a real possi...","ESI em mínimas, queda de 1,50 a 2,50. BK é uma...",esi mínimas queda 150 250 bk possibilidade real
2,"For the last quarter of 2010 , Componenta 's n...",positive,"For the last quarter of 2010 , Componenta 's n...","No último trimestre de 2010, as vendas líquida...",último trimestre 2010 vendas líquidas componen...
3,According to the Finnish-Russian Chamber of Co...,neutral,According to the Finnish-Russian Chamber of Co...,"Segundo a Câmara de Comércio Finlandesa-Russa,...",segundo câmara comércio finlandesarussa todas ...
4,The Swedish buyout firm has sold its remaining...,neutral,The Swedish buyout firm has sold its remaining...,A empresa sueca de aquisições vendeu sua parti...,empresa sueca aquisições vendeu participação r...


## Treinando Modelo Naive Bayes com CountVectorizer

In [6]:
# Seed para reprodutibilidade
np.random.seed(42)

In [7]:
# Dividindo os dados em treino e teste
x = df['sentence_pt_clean']
y = df['sentiment']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y)

In [8]:
# GridSearchCV para otimização de hiperparâmetros do modelo Naive Bayes com CountVectorizer
param_grid = {
    'countvectorizer__max_features': [1000, 2000, 5000],
    'countvectorizer__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'multinomialnb__alpha': [0.01, 0.1, 1.0]
}

pipeline = make_pipeline(CountVectorizer(), MultinomialNB())

model_count = GridSearchCV(
    pipeline,
    param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1)

model_count.fit(x_train, y_train)

print(f"Best parameters: {model_count.best_params_}")
print(f"Best CV score: {model_count.best_score_:.4f}")

y_pred = model_count.predict(x_test)

print(classification_report(y_test, y_pred))

Best parameters: {'countvectorizer__max_features': 5000, 'countvectorizer__ngram_range': (1, 2), 'multinomialnb__alpha': 1.0}
Best CV score: 0.6162
              precision    recall  f1-score   support

    negative       0.38      0.45      0.41       172
     neutral       0.76      0.74      0.75       626
    positive       0.70      0.67      0.69       371

    accuracy                           0.68      1169
   macro avg       0.62      0.62      0.62      1169
weighted avg       0.69      0.68      0.68      1169



In [10]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 2, 5],
    'tfidf__sublinear_tf': [True, False],
    'clf__alpha': [0.1, 0.5, 1.0]  # suavização do Naive Bayes
}

model_tfidf = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)

model_tfidf.fit(x_train, y_train)

print(f"Best parameters: {model_tfidf.best_params_}")
print(f"Best CV score: {model_tfidf.best_score_:.4f}")

y_pred_tfidf = model_tfidf.predict(x_test)

print(classification_report(y_test, y_pred_tfidf))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'clf__alpha': 0.1, 'tfidf__max_features': 3000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
Best CV score: 0.5883
              precision    recall  f1-score   support

    negative       0.41      0.36      0.38       172
     neutral       0.72      0.79      0.76       626
    positive       0.70      0.63      0.66       371

    accuracy                           0.68      1169
   macro avg       0.61      0.59      0.60      1169
weighted avg       0.67      0.68      0.67      1169



In [11]:
# Calcular métricas para ambos
print("=" * 70)
print("COMPARAÇÃO: CountVectorizer vs TfidfVectorizer")
print("=" * 70)

for name, model, y_pred in [("CountVectorizer", model_count, y_pred), 
                             ("TfidfVectorizer", model_tfidf, y_pred_tfidf)]:
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    print(f"\n{name}:")
    print(f"  Accuracy:  {acc:2.2%}")
    print(f"  Precision: {prec:2.2%}")
    print(f"  Recall:    {rec:2.2%}")
    print(f"  F1 Score:  {f1:2.2%}")

print("=" * 70)

COMPARAÇÃO: CountVectorizer vs TfidfVectorizer

CountVectorizer:
  Accuracy:  67.75%
  Precision: 68.69%
  Recall:    67.75%
  F1 Score:  68.16%

TfidfVectorizer:
  Accuracy:  67.66%
  Precision: 67.04%
  Recall:    67.66%
  F1 Score:  67.18%
